## Join the DataFrames
In the next two chapters you'll be working to build a model that predicts whether or not a flight will be delayed based on the flights data we've been working with. This model will also include information about the plane that flew the route, so the first step is to join the two tables: flights and planes!

- First, rename the year column of planes to plane_year to avoid duplicate column names.
- Create a new DataFrame called model_data by joining the flights table with planes using the tailnum column as the key.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5bb29c12-0967-49ce-b02d-6165c1050f1c.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739216953753).
SparkSession available as 'spark'.
# Rename year column
planes = planes.withColumnRenamed('year', 'plane_year')

# Join the DataFrames
model_data = flights.join(planes, on='tailnum', how="leftouter")

## String to integer
Now you'll use the .cast() method you learned in the previous exercise to convert all the appropriate columns from your DataFrame model_data to integers!

To convert the type of a column using the .cast() method, you can write code like this:

dataframe = dataframe.withColumn("col", dataframe.col.cast("new_type"))

Use the method .withColumn() to .cast() the following columns to type "integer". Access the columns using the df.col notation:
- model_data.arr_delay
- model_data.air_time
- model_data.month
- model_data.plane_year

In [ ]:
# Cast the columns to integers
model_data = model_data.withColumn("arr_delay", model_data.arr_delay.cast("integer"))
model_data = model_data.withColumn("air_time", model_data.air_time.cast("integer"))
model_data = model_data.withColumn("month", model_data.month.cast("integer"))
model_data = model_data.withColumn("plane_year", model_data.plane_year.cast("integer"))

## Create a new column
In the last exercise, you converted the column plane_year to an integer. This column holds the year each plane was manufactured. However, your model will use the planes' age, which is slightly different from the year it was made!

Create the column plane_age using the .withColumn() method and subtracting the year of manufacture (column plane_year) from the year (column year) of the flight.

In [ ]:
# Create the column plane_age
model_data = model_data.withColumn("plane_age", model_data.year - model_data.plane_year)

## Making a Boolean
Consider that you're modeling a yes or no question: is the flight late? However, your data contains the arrival delay in minutes for each flight. Thus, you'll need to create a boolean column which indicates whether the flight was late or not!

- Use the .withColumn() method to create the column is_late. This column is equal to model_data.arr_delay > 0.
- Convert this column to an integer column so that you can use it in your model and name it label (this is the default name for the response variable in Spark's machine learning routines).
- Filter out missing values (this has been done for you).

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224307892).
SparkSession available as 'spark'.
# Create is_late
model_data = model_data.withColumn("is_late", model_data.arr_delay > 0)

# Convert to an integer
model_data = model_data.withColumn("label", model_data.is_late.cast("integer"))

# Remove missing values
model_data = model_data.filter("arr_delay is not NULL and dep_delay is not NULL and air_time is not NULL and plane_year is not NULL")

## Carrier
In this exercise you'll create a StringIndexer and a OneHotEncoder to code the carrier column. To do this, you'll call the class constructors with the arguments inputCol and outputCol.

The inputCol is the name of the column you want to index or encode, and the outputCol is the name of the new column that the Transformer should create.

- Create a StringIndexer called carr_indexer by calling StringIndexer() with inputCol="carrier" and outputCol="carrier_index".
- Create a OneHotEncoder called carr_encoder by calling OneHotEncoder() with inputCol="carrier_index" and outputCol="carrier_fact".

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224456460).
SparkSession available as 'spark'.
# Create a StringIndexer
carr_indexer = StringIndexer(inputCol="carrier", outputCol="carrier_index")

# Create a OneHotEncoder
carr_encoder = OneHotEncoder(inputCol="carrier_index", outputCol="carrier_fact")

## Destination
Now you'll encode the dest column just like you did in the previous exercise.

- Create a StringIndexer called dest_indexer by calling StringIndexer() with inputCol="dest" and outputCol="dest_index".
- Create a OneHotEncoder called dest_encoder by calling OneHotEncoder() with inputCol="dest_index" and outputCol="dest_fact".

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224552359).
SparkSession available as 'spark'.

# Create a StringIndexer
dest_indexer = StringIndexer(inputCol="dest", outputCol="dest_index")

# Create a OneHotEncoder
dest_encoder = OneHotEncoder(inputCol="dest_index", outputCol="dest_fact")
ERROR! Session/line number was not unique in database. History logging moved to new session 8

## Assemble a vector
The last step in the Pipeline is to combine all of the columns containing our features into a single column. This has to be done before modeling can take place because every Spark modeling routine expects the data to be in this form. You can do this by storing each of the values from a column as an entry in a vector. Then, from the model's point of view, every observation is a vector that contains all of the information about it and a label that tells the modeler what value that observation corresponds to.

Because of this, the pyspark.ml.feature submodule contains a class called VectorAssembler. This Transformer takes all of the columns you specify and combines them into a new vector column.

- Create a VectorAssembler by calling VectorAssembler() with the inputCols names as a list and the outputCol name "features".
- The list of columns should be ["month", "air_time", "carrier_fact", "dest_fact", "plane_age"].

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224667790).
SparkSession available as 'spark'.

# Make a VectorAssembler
vec_assembler = VectorAssembler(inputCols=["month", "air_time", "carrier_fact", "dest_fact", "plane_age"], outputCol="features")

## Create the pipeline
You're finally ready to create a Pipeline!

Pipeline is a class in the pyspark.ml module that combines all the Estimators and Transformers that you've already created. This lets you reuse the same modeling process over and over again by wrapping it up in one simple object. Neat, right?

- Import Pipeline from pyspark.ml.
- Call the Pipeline() constructor with the keyword argument stages to create a Pipeline called flights_pipe.
    - stages should be a list holding all the stages you want your data to go through in the pipeline. Here this is just: [dest_indexer, dest_encoder, carr_indexer, carr_encoder, vec_assembler]

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224774104).
SparkSession available as 'spark'.
# Import Pipeline
from pyspark.ml import Pipeline

# Make the pipeline
flights_pipe = Pipeline(stages=[dest_indexer, dest_encoder, carr_indexer, carr_encoder, vec_assembler])

## Transform the data
Hooray, now you're finally ready to pass your data through the Pipeline you created!

Create the DataFrame piped_data by calling the Pipeline methods .fit() and .transform() in a chain. Both of these methods take model_data as their only argument.



In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224896184).
SparkSession available as 'spark'.
# Fit and transform the data
piped_data = flights_pipe.fit(model_data).transform(model_data)

## Split the data
Now that you've done all your manipulations, the last step before modeling is to split the data!

Use the DataFrame method .randomSplit() to split piped_data into two pieces, training with 60% of the data, and test with 40% of the data by passing the list [.6, .4] to the .randomSplit() method.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739224969466).
SparkSession available as 'spark'.
# Split the data into training and test sets
training, test = piped_data.randomSplit([.6, .4])

## Create the modeler
The Estimator you'll be using is a LogisticRegression from the pyspark.ml.classification submodule.

- Import the LogisticRegression class from pyspark.ml.classification.
- Create a LogisticRegression called lr by calling LogisticRegression() with no arguments.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225081833).
SparkSession available as 'spark'.
# Import LogisticRegression
from pyspark.ml.classification import LogisticRegression

# Create a LogisticRegression Estimator
lr = LogisticRegression()

## Create the evaluator
The first thing you need when doing cross validation for model selection is a way to compare different models. Luckily, the pyspark.ml.evaluation submodule has classes for evaluating different kinds of models. Your model is a binary classification model, so you'll be using the BinaryClassificationEvaluator from the pyspark.ml.evaluation module.

This evaluator calculates the area under the ROC. This is a metric that combines the two kinds of errors a binary classifier can make (false positives and false negatives) into a simple number. You'll learn more about this towards the end of the chapter!

- Import the submodule pyspark.ml.evaluation as evals.
- Create evaluator by calling evals.BinaryClassificationEvaluator() with the argument metricName="areaUnderROC".

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225204650).
SparkSession available as 'spark'.
# Import the evaluation submodule
import pyspark.ml.evaluation as evals

# Create a BinaryClassificationEvaluator
evaluator = evals.BinaryClassificationEvaluator(metricName="areaUnderROC")

## Make a grid
Next, you need to create a grid of values to search over when looking for the optimal hyperparameters. The submodule pyspark.ml.tuning includes a class called ParamGridBuilder that does just that (maybe you're starting to notice a pattern here; PySpark has a submodule for just about everything!).

You'll need to use the .addGrid() and .build() methods to create a grid that you can use for cross validation. The .addGrid() method takes a model parameter (an attribute of the model Estimator, lr, that you created a few exercises ago) and a list of values that you want to try. The .build() method takes no arguments, it just returns the grid that you'll use later.

- Import the submodule pyspark.ml.tuning under the alias tune.
- Call the class constructor ParamGridBuilder() with no arguments. Save this as grid.
- Call the .addGrid() method on grid with lr.regParam as the first argument and np.arange(0, .1, .01) as the second argument. This second call is a function from the numpy module (imported as np) that creates a list of numbers from 0 to .1, incrementing by .01. Overwrite grid with the result.
- Update grid again by calling the .addGrid() method a second time create a grid for lr.elasticNetParam that includes only the values [0, 1].
- Call the .build() method on grid and overwrite it with the output.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225274802).
SparkSession available as 'spark'.
# Import the tuning submodule
import pyspark.ml.tuning as tune

# Create the parameter grid
grid = tune.ParamGridBuilder()

# Add the hyperparameter
grid = grid.addGrid(lr.regParam, np.arange(0, .1, .01))
grid = grid.addGrid(lr.elasticNetParam, [0, 1])

# Build the grid
grid = grid.build()

## Make the validator
The submodule pyspark.ml.tuning also has a class called CrossValidator for performing cross validation. This Estimator takes the modeler you want to fit, the grid of hyperparameters you created, and the evaluator you want to use to compare your models.

The submodule pyspark.ml.tune has already been imported as tune. You'll create the CrossValidator by passing it the logistic regression Estimator lr, the parameter grid, and the evaluator you created in the previous exercises.

- Create a CrossValidator by calling tune.CrossValidator() with the arguments:
    - estimator=lr
    - estimatorParamMaps=grid
    - evaluator=evaluator
- Name this object cv.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225531251).
SparkSession available as 'spark'.
# Create the CrossValidator
cv = tune.CrossValidator(estimator=lr,
               estimatorParamMaps=grid,
               evaluator=evaluator
               )

## Fit the model(s)
You're finally ready to fit the models and select the best one!

Unfortunately, cross validation is a very computationally intensive procedure. Fitting all the models would take too long on DataCamp.

To do this locally you would use the code:

# Fit cross validation models
models = cv.fit(training)

# Extract the best model
best_lr = models.bestModel

Remember, the training data is called training and you're using lr to fit a logistic regression model. Cross validation selected the parameter values regParam=0 and elasticNetParam=0 as being the best. These are the default values, so you don't need to do anything else with lr before fitting the model.

- Create best_lr by calling lr.fit() on the training data.
- Print best_lr to verify that it's an object of the LogisticRegressionModel class.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225617905).
SparkSession available as 'spark'.
# Call lr.fit()
best_lr = lr.fit(training)

# Print best_lr
print(best_lr)
LogisticRegressionModel: uid=LogisticRegression_502c7105f230, numClasses=2, numFeatures=83

## Evaluate the model
Remember the test data that you set aside waaaaaay back in chapter 3? It's finally time to test your model on it! You can use the same evaluator you made to fit the model.

- Use your model to generate predictions by applying best_lr.transform() to the test data. Save this as test_results.
- Call evaluator.evaluate() on test_results to compute the AUC. Print the output.

In [ ]:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.2.0
      /_/

Using Python version 3.9.7 (default, Sep 10 2021 00:03:59)
Spark context Web UI available at http://5feae156-8cd7-4838-a9c3-cdb4e09f19b0.sessions.sessions.svc.cluster.local:4040
Spark context available as 'sc' (master = local[*], app id = local-1739225708406).
SparkSession available as 'spark'.
# Use the model to predict the test set
test_results = best_lr.transform(test)

# Evaluate the predictions
print(evaluator.evaluate(test_results))
0.7123313100891033